The goal of this notebook is to show examples of how to work with data stored in some of the most common data types in Atmospheric Science. Namely: NetCDF, CSV, and ICARTT files. So that you can start working with them yourself in the Workshop_4_Exercises notebook.

In [ ]:
###### IMPORTS GO HERE ######
# PUBLIC LIBRARIES #
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

The first type of file we're going to look at is CSV files, these are commonly used to store all kinds of data, typically though the data in a CSV file is 1-D time series data. CSV stands for Comma Separated Values, this just means each line in the file is a value followed by a comma and then the next value and so on. There are multiple ways to read in CSV files, I will only showcase a few.

In [ ]:
# Often the data we work with is on our machine so we'll have to use the path from
# our code or our terminal to the file in order to pull it in.
# The file path below points to a small subset of data for Christman Field
cf_file_path = './Workshop_4_Data_Files/ftc01.csv'

In [ ]:
# There are multiple ways to open CSV files, but today we're going to focus on Pandas
# Pandas is designed to work with CSV files in a human friendly way while also being efficient
# To load in a csv file we just use the .read_csv() method from Pandas
# We just need to input the path to the file/name of the file, as well as information
# which helps Pandas determine what's data versus what is metadata.
# The 'header = [0,1]' code tells Pandas that the first two lines of the file describe the 
# data stored in the rest of the file
csv_example = pd.read_csv(cf_file_path,header = [0,1])

In [ ]:
# Once we've loaded the data with Pandas we can examine as we would any other Pandas dataframe
csv_example.head()

In [ ]:
csv_example.tail()

In [ ]:
csv_example.describe()

In [ ]:
# Dates are often stores as strings, not datetime objects like we'd prefer
# This code let's us put the data into a usable format for plotting
# by converting the strings to datetime objects
csv_example['Date and Time'] = csv_example['Date and Time'].apply(pd.to_datetime)

In [ ]:
# We can also plot the data as we would any other Panda's dataframe
# Make the base for the plot
fig,ax = plt.subplots(2,1,figsize = (9,8))
# Plot the data
ax[0].plot(csv_example['Date and Time'],csv_example['Air Temp'],color = 'k')
ax[1].plot(csv_example['Date and Time'],csv_example['Solar Rad'],color = 'k')
# Label the plots
ax[0].set_title('Air Temperature')
ax[1].set_title('Solar Irradiance')

ax[0].set_ylabel('Temperature [$^o$F]')
ax[1].set_ylabel('Solar Irradiance [W/m$^2$]')

ax[0].set_xlabel('Time')
ax[1].set_xlabel('Time')
# Fix the spacing so they look better
plt.subplots_adjust(hspace = 0.275)

plt.show()

In [ ]:
# If we wanted to make this plot but only for time periods where the sun is down
nighttime_df = csv_example.loc[(csv_example['Solar Rad'] == 0).values]
daytime_df = csv_example.loc[(csv_example['Solar Rad'] > 0).values]
# This exact same technique can be used to clean dataframes of bad values
# by limiting it to just the rows that meet a certain condition

In [ ]:
fig,ax = plt.subplots(2,1,figsize = (10,8))

ax[0].scatter(nighttime_df['Date and Time'],nighttime_df['Air Temp'],s = 10, c= 'k')
ax[1].scatter(daytime_df['Date and Time'],daytime_df['Air Temp'],s = 10, c ='k')

ax[0].set_title('Nighttime Air Temperature')
ax[1].set_title('Daytime Air Temperature')

ax[0].set_ylabel('Temperature [$^o$F]')
ax[1].set_ylabel('Temperature [$^o$F]')

ax[0].set_xlabel('Time')
ax[1].set_xlabel('Time')

plt.subplots_adjust(hspace = 0.275)

plt.show()

Next we're going to learn how to open netCDF files. Fortunately this is just as easy as CSVs if we use XArray!

In [ ]:
# This is a file path leading to a small piece of HadISST Global SST Data
# This data is global mean SST temperatures for every month of 2023
netcdf_filepath = './Workshop_4_Data_Files/HadISST_2023_Subset_Global.nc'

In [ ]:
# We can work with this like we would any XArray DataSet
# We use .open_dataset() to open up the file
netcdf_example = xr.open_dataset(netcdf_filepath)

In [ ]:
# We can examine the Dataet by calling its name
netcdf_example

In [ ]:
# We can do operations on the data
# This line gets the mean for the entirety of 2023
# it works by taking amean across the entire dimension labeled 'time'
# You can do the same thing across longitudes/latitudes as well
yearly_mean = netcdf_example.mean(dim = 'time')
# The code below plots the data we created in the line above
# Make the base for the figure
fig,ax = plt.subplots(1,1,figsize = (8,5),subplot_kw = {'projection':ccrs.PlateCarree()})
# Make a colormesh of the SST data
pc0 = ax.pcolormesh(yearly_mean['longitude'],yearly_mean['latitude'],yearly_mean['sst'],cmap = 'turbo')
# Add coastlines
ax.coastlines()
# Change the background of the plot to grey so land looks better
ax.set_facecolor('grey')
# Add a title and colorbar
ax.set_title('2023 Yearly Mean SST from HadISST')
plt.colorbar(pc0,label = 'SST [$^o$C]')
plt.show()

The plot should look wrong, we can see that the values near the poles where there is abundant sea-ice seem to be the problem, this is because we never cleaned the data. This is an important step in any process. So let's learn how to clean data.

If we examine the cell above we can see that the data does not have a particular value corresponding to bad data so we need to manually examine the data.

Let's plot a histogram of the data to see the distribution of values and figure out our threshold.

In [ ]:
# This makes a simple histogram of the data showing its distribution
plt.hist(np.array(yearly_mean['sst']).flatten(),bins = np.arange(-700,75,25),color = 'k',rwidth = 0.8)
plt.ylabel('Number of Gridcells with Specified SST Value')
plt.xlabel('SST [$^o$C]')
plt.xticks(np.arange(-700,75,100))
plt.show()

This chart tells us the majority of the data is in the physically realistic range with a few random values less than -50 celsius. Let's turn all values less then -50 to a value called NaN which stands for Not A Number. This is a special value which tells lots of code to ignore it, including matplotlib.

In [ ]:
# This line tells xarray to take our SST data, and keep all of the values
# of SST that are greater than -50 degrees Celsius, all others are set to NaN
netcdf_example['sst'] = netcdf_example['sst'].where(netcdf_example['sst'] > -50)

Replotting the data shows our filtering to have worked.

In [ ]:
# Replotting the data shows this to be true
yearly_mean = netcdf_example.mean(dim = 'time')
# The code below plots the data as we did a few cells above
fig,ax = plt.subplots(1,1,figsize = (8,5),subplot_kw = {'projection':ccrs.PlateCarree()})
pc0 = ax.pcolormesh(yearly_mean['longitude'],yearly_mean['latitude'],yearly_mean['sst'],cmap = 'turbo')
ax.coastlines()
ax.set_facecolor('grey')
ax.set_title('2023 Yearly Mean SST from HadISST')
plt.colorbar(pc0,label = 'SST [$^o$C]')
plt.show()

In [ ]:
# This plots the global mean SST for each month of 2023
# Why do you think February is the hottest month for SST?
plt.plot(netcdf_example['sst'].mean(dim = ['latitude','longitude'],skipna = True))
plt.ylabel('SST [$^o$C]')
plt.xlabel('Month')
plt.xticks(np.arange(0,12,1),['J','F','M','A','M','J','J','A','S','O','N','D'])
plt.title('Monthly Global Mean SST')
plt.show()

Now let's look at ICARTT files. These are common in the air quality field, especially with aircraft measurements. You will probably work with them in ATS 621.

ICARTT files are functionally CSV's with a very formalized header which describes all of the metadata that went into making the dataset.

[here is a link to how the headers for an ICARTT file work](https://www-air.larc.nasa.gov/missions/etc/IcarttDataFormat.htm)

Because the header is specified we can make use of it.

In [ ]:
# First let's get a path settled for the file
ICARTT_filepath = './Workshop_4_Data_Files/WS4_Tutorial_WECAN_Reprocessed.ict'

In [ ]:
# Below are some functions I wrote that read the header of an ICARTT file and use
# It to make a PANDAS Dataframe
def get_ICT_header_length(ICT_file:str) -> int:
    '''
        Grabs the length of the header for the ICARTT file of interest
        and returns it so this information can be used by other functions.

        Parameters:
            ICT_file (str): The name/filepath to the .ict file
        
        Returns:
            Header_length (int): The length of the header in lines
    '''

    # Stage the file for Python's I/O to read it
    opened_file = open(ICT_file)
    # Bring in the first line, and grab the length of the header
    header_length = int(opened_file.readline().split(',')[0])
    # Close the file so it doesn't stick around
    opened_file.close()

    return header_length

def open_ICT_as_Dataframe(ICT_file:str) -> pd.DataFrame:
    '''
        Opens an ICARTT file (.ict) and uses some information from the
        header to convert it into a Pandas Dataframe.

        Parameters:
            ICT_file (str): The name/filepath of the .ict file
        
        Returns:
            ict_df (pd.DataFrame): The data within the .ict file as a DataFrame
    '''

    ict_df = pd.read_csv(ICT_file,skiprows = get_ICT_header_length(ICT_file)-1)

    return ict_df

In [ ]:
# This opens up the ICARTT file as a pandas dataframe using the functions above
ICARTT_df = open_ICT_as_Dataframe(ICARTT_filepath)

In [ ]:
ICARTT_df

In [ ]:
# This is just a simple plot to see the plane's height 
# in pressure coordinates over the course of the flight
fig,ax = plt.subplots(1,1,figsize = (8,5))

ax.plot(ICARTT_df['UTC_mid'],ICARTT_df['PRESSURE'])
ax.set_ylabel('Pressure [hPa]')
ax.set_ylim(950,400)
ax.set_xlabel('UTC Midtimestep [s]')
ax.set_title('Plane Pressure Level Over Flight')

plt.show()

Other than having to open up the file differently, ICARTT files are similar to CSVs, they just include lots of metadata that informs you about the data you're using

### Now that you've seen working with files really isn't so bad let's move to the exercises for Workshop 4.